In [ ]:
import importlib
import sys


def reimport_trackers():
    """Force-reload trackers after a fresh pip install (Colab-friendly)."""
    mods = [k for k in sys.modules if k.startswith("trackers")]
    for name in sorted(mods, reverse=True):
        importlib.reload(sys.modules[name])
    print(f"Reloaded {len(mods)} trackers module(s).")

# Fine-tune OSNet on MOT17

Fine-tune the default **OSNet (MSMT17)** encoder on MOT17 train-half GT crops, then
measure whether the in-domain adapter improves ReID quality and downstream tracking.

**Pipeline**

1. Download MOT17 train/val (Trackers CLI) + YOLOX val detections (gdown zip).
2. Crop **train-sequence** train-half pedestrians and fine-tune OSNet on **all** train identities.
3. Benchmark **retrieval** on **val-sequence** GT crops (mAP / Rank-1) — disjoint people from training.
4. Benchmark **tracking** on the val-half split with BoT-SORT + ReID (HOTA / IDF1).

**Split safety:** crops use frames `1 … L/2` only; tracking eval uses the val-half
frames + YOLOX detections (BoT-SORT / ByteTrack protocol). No Google Drive required.

> **Runtime:** `Runtime → Change runtime type → T4 GPU` before running.

---

## 1. Install

Install Trackers with the `[reid]` extra plus notebook dependencies. **Run this cell before §2 Setup.**


In [ ]:
import subprocess

TRACKERS_SPEC = "trackers[reid] @ git+https://github.com/roboflow/trackers.git@feat/core/reid-training"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", TRACKERS_SPEC])
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "supervision",
        "opencv-python",
        "gdown",
        "matplotlib",
        "scikit-learn",
    ]
)

reimport_trackers()

In [ ]:
import warnings

import torch

warnings.filterwarnings("ignore")

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  |  Device: {device}")

## 2. Setup

Import the ReID training API, define paths, and add **notebook-local** helpers
for YOLOX detections and embedding plots.

Only `load_mot_file` is imported from `trackers` — YOLOX I/O is defined below.
On Colab, data and outputs live under `/content/`.


In [ ]:
import subprocess
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import supervision as sv
from sklearn.decomposition import PCA

# Only load_mot_file from trackers — YOLOX det I/O is notebook-local (load_yolox_val_dets below).
from trackers import load_mot_file
from trackers.core.reid import ReIDEvaluator, ReIDModel
from trackers.core.reid.training import (
    TrainConfig,
    build_retrieval_split,
    generate_mot_patches,
    train_reid,
)
from trackers.eval import box_iou

try:
    import google.colab  # noqa: F401

    REPO_ROOT = Path("/content")
except ImportError:
    REPO_ROOT = Path("..").resolve()
OUT = REPO_ROOT / "outputs"
MOT17_TRAIN = REPO_ROOT / "mot17" / "train"
MOT17_VAL = REPO_ROOT / "mot17" / "val"
YOLOX_ROOT = REPO_ROOT / "MOT17_yolox_dets"
YOLOX_VAL_DIR = YOLOX_ROOT / "val"
YOLOX_FILE_ID = "1BuXtPWf8QbPU_y1i2xY2IbTE-rj3l6qT"
PATCH_DIR = OUT / "reid_mot17_crops_train"
PATCH_VAL_DIR = OUT / "reid_mot17_crops_val"
CHECKPOINT_DIR = OUT / "reid_mot17_osnet"
TRACK_OUT = OUT / "trackers_reid_outputs"

VAL_SEQS = [
    "MOT17-02-FRCNN",
    "MOT17-04-FRCNN",
    "MOT17-05-FRCNN",
    "MOT17-09-FRCNN",
    "MOT17-10-FRCNN",
    "MOT17-11-FRCNN",
    "MOT17-13-FRCNN",
]


def yolox_det_path(seq: str) -> Path:
    return YOLOX_VAL_DIR / f"{seq.replace('-FRCNN', '')}_val.txt"


def yolox_det_frame_offset(det_path: Path) -> int:
    """YOLOX val dets use absolute frame ids (302+); val-half GT/images use 1..N."""
    min_frame = None
    with det_path.open() as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame = int(float(parts[0]))
            min_frame = frame if min_frame is None else min(min_frame, frame)
    return (min_frame - 1) if min_frame and min_frame > 1 else 0


def load_yolox_val_dets(det_path: Path, frame_offset: int = 0) -> dict[int, sv.Detections]:
    """Load YOLOX export: frame,x1,y1,x2,y2,score → {frame: Detections}."""
    frame_map: dict[int, list[list[float]]] = {}
    with det_path.open() as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame = int(float(parts[0])) - frame_offset
            if frame < 1:
                continue
            x1, y1, x2, y2, score = map(float, parts[1:6])
            if score <= 0:
                continue
            frame_map.setdefault(frame, []).append([x1, y1, x2, y2, score])

    return {
        frame: sv.Detections(
            xyxy=np.array(boxes, dtype=np.float32)[:, :4],
            confidence=np.array(boxes, dtype=np.float32)[:, 4],
        )
        for frame, boxes in frame_map.items()
    }


def load_val_dets(det_path: Path) -> dict[int, sv.Detections]:
    """YOLOX dets with frame ids remapped to match val-half images."""
    return load_yolox_val_dets(det_path, frame_offset=yolox_det_frame_offset(det_path))


def match_det_boxes_to_gt_ids(gt_frame, det_xyxy: np.ndarray, min_iou: float = 0.5) -> np.ndarray:
    gt_xyxy = sv.xywh_to_xyxy(gt_frame.boxes)
    keep = (gt_frame.confidences > 0) & (gt_frame.classes == 1)
    gt_xyxy, gt_ids = gt_xyxy[keep], gt_frame.ids[keep]
    if len(gt_xyxy) == 0:
        return np.full(len(det_xyxy), -1, dtype=np.int64)
    ious = box_iou(det_xyxy.astype(np.float64), gt_xyxy.astype(np.float64))
    assigned = np.full(len(det_xyxy), -1, dtype=np.int64)
    for i in range(len(det_xyxy)):
        j = int(np.argmax(ious[i]))
        if ious[i, j] >= min_iou:
            assigned[i] = int(gt_ids[j])
    return assigned


def collect_yolox_gt_embeddings(
    model: ReIDModel,
    sequences: list[str],
    *,
    frame_stride: int = 5,
    max_frames_per_seq: int | None = None,
    min_det_conf: float = 0.5,
) -> tuple[np.ndarray, np.ndarray]:
    """Embed YOLOX detections matched to GT track ids (tracking-domain crops)."""
    embeddings, gt_ids = [], []
    for seq in sequences:
        gt_by_frame = load_mot_file(MOT17_VAL / seq / "gt" / "gt.txt")
        det_by_frame = load_val_dets(yolox_det_path(seq))
        frame_paths = sorted((MOT17_VAL / seq / "img1").glob("*.jpg"))
        frame_indices = list(range(1, len(frame_paths) + 1, frame_stride))
        if max_frames_per_seq is not None:
            frame_indices = frame_indices[:max_frames_per_seq]

        for frame_idx in frame_indices:
            dets = det_by_frame.get(frame_idx)
            gt_frame = gt_by_frame.get(frame_idx)
            if dets is None or len(dets) == 0 or gt_frame is None:
                continue
            dets = dets[dets.confidence >= min_det_conf]
            if len(dets) == 0:
                continue

            bgr = cv2.imread(str(frame_paths[frame_idx - 1]))
            matched = match_det_boxes_to_gt_ids(gt_frame, dets.xyxy)
            feats = model.extract_features(dets, bgr)
            for i in range(len(dets)):
                if matched[i] < 0:
                    continue
                embeddings.append(feats[i])
                gt_ids.append(int(matched[i]))

    if not embeddings:
        raise RuntimeError("No GT-matched YOLOX embeddings collected — run §3 downloads first.")
    return np.stack(embeddings), np.array(gt_ids, dtype=np.int64)


def sample_intra_inter_cosine_distances(
    embeddings: np.ndarray,
    gt_ids: np.ndarray,
    *,
    n_intra: int = 8000,
    n_inter: int = 8000,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """Sample cosine distances for same-GT-id pairs vs different-GT-id pairs."""
    normed = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12)
    rng = np.random.default_rng(seed)
    by_id: dict[int, list[int]] = {}
    for idx, pid in enumerate(gt_ids):
        by_id.setdefault(int(pid), []).append(idx)

    ids_with_pair = [pid for pid, idxs in by_id.items() if len(idxs) >= 2]
    all_ids = list(by_id.keys())
    if not ids_with_pair:
        raise ValueError("Need at least one GT id with two embeddings for intra-class distances.")

    intra = np.empty(n_intra)
    for k in range(n_intra):
        pid = rng.choice(ids_with_pair)
        i, j = rng.choice(by_id[pid], size=2, replace=False)
        intra[k] = 1.0 - float(normed[i] @ normed[j])

    inter = np.empty(n_inter)
    for k in range(n_inter):
        pid_a, pid_b = rng.choice(all_ids, size=2, replace=False)
        i = rng.choice(by_id[pid_a])
        j = rng.choice(by_id[pid_b])
        inter[k] = 1.0 - float(normed[i] @ normed[j])

    return intra, inter


def suggest_emb_dist_threshold(intra: np.ndarray, inter: np.ndarray) -> float:
    """Heuristic θ_emb from BoT-SORT-style distance histograms (§8c → §9)."""
    lo = float(np.quantile(intra, 0.90))
    hi = float(np.quantile(inter, 0.10))
    return (lo + hi) / 2


def plot_intra_inter_distance_histogram(
    intra: np.ndarray,
    inter: np.ndarray,
    *,
    title: str,
    ax: plt.Axes | None = None,
    emb_dist_threshold: float | None = None,
) -> plt.Axes:
    """BoT-SORT-style histogram of embedding distances (same id vs different id)."""
    ax = ax or plt.gca()
    bins = np.linspace(0, 1.2, 50)
    ax.hist(intra, bins=bins, alpha=0.55, density=True, label="Same GT ID", color="#3366CC")
    ax.hist(inter, bins=bins, alpha=0.55, density=True, label="Different GT ID", color="#DC3912")
    if emb_dist_threshold is not None:
        ax.axvline(
            emb_dist_threshold,
            color="black",
            ls="--",
            lw=1.5,
            label=f"θ_emb={emb_dist_threshold:.2f}",
        )
    ax.set(xlabel="Cosine distance", ylabel="Density", title=title)
    ax.legend()
    ax.grid(True, alpha=0.25)
    return ax


def plot_pca_side_by_side(
    emb_left: np.ndarray,
    emb_right: np.ndarray,
    labels: np.ndarray,
    *,
    left_title: str,
    right_title: str,
    suptitle: str,
) -> None:
    """Side-by-side 2D PCA scatter; same color = same identity label."""
    coords_l = PCA(n_components=2, random_state=0).fit_transform(emb_left)
    coords_r = PCA(n_components=2, random_state=0).fit_transform(emb_right)
    unique = np.unique(labels)
    cmap = plt.colormaps["tab20"].resampled(max(len(unique), 1))
    colors = {pid: cmap(i % 20) for i, pid in enumerate(unique)}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
    for ax, coords, title in [(axes[0], coords_l, left_title), (axes[1], coords_r, right_title)]:
        for pid in unique:
            mask = labels == pid
            ax.scatter(coords[mask, 0], coords[mask, 1], s=26, alpha=0.85, color=colors[pid])
        ax.set(title=title, xlabel="PC1")
        ax.grid(True, alpha=0.25)
    axes[0].set_ylabel("PC2")
    fig.suptitle(suptitle, y=1.02)
    plt.tight_layout()
    plt.show()


print("Setup complete.")

---
## 3. Download data

### 3a. MOT17 frames and ground truth

Downloads train + val into `mot17/`. Train sequences supply GT crops; val sequences
supply GT + images for the tracking benchmark.

In [ ]:
FORCE_DOWNLOAD_MOT17 = False

train_seqs = sum(1 for _ in MOT17_TRAIN.glob("MOT17-*/gt/gt.txt"))
mot17_ready = train_seqs >= len(VAL_SEQS) and all((MOT17_VAL / seq / "gt" / "gt.txt").is_file() for seq in VAL_SEQS)

if FORCE_DOWNLOAD_MOT17 or not mot17_ready:
    subprocess.run(
        [
            "trackers",
            "download",
            "mot17",
            "--split",
            "train,val",
            "--asset",
            "annotations,frames",
            "-o",
            str(REPO_ROOT),
        ],
        check=True,
    )
else:
    print("MOT17 already present.")

train_seqs = sum(1 for _ in MOT17_TRAIN.glob("MOT17-*/gt/gt.txt"))
print(f"Train sequences ready: {train_seqs}/{len(VAL_SEQS)}")

### 3b. YOLOX val detections

Public YOLOX detector outputs for the MOT17 val-half split (same zip used in
BoT-SORT / ByteTrack eval notebooks). Extracted to `MOT17_yolox_dets/val/`.

In [ ]:
import zipfile

import gdown

FORCE_DOWNLOAD_YOLOX = False
zip_path = YOLOX_ROOT / "yolox_detections_MOT17.zip"

yolox_ready = YOLOX_VAL_DIR.is_dir() and len(list(YOLOX_VAL_DIR.glob("MOT17-*_val.txt"))) >= len(VAL_SEQS)

if FORCE_DOWNLOAD_YOLOX or not yolox_ready:
    YOLOX_ROOT.mkdir(parents=True, exist_ok=True)
    gdown.download(id=YOLOX_FILE_ID, output=str(zip_path), quiet=False)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(YOLOX_ROOT)
else:
    print("YOLOX detections already present.")

n_dets = len(list(YOLOX_VAL_DIR.glob("MOT17-*_val.txt")))
print(f"YOLOX val det files: {n_dets}/{len(VAL_SEQS)}")

---
## 4. Generate GT crops

Two crop roots — **train sequences** for fine-tuning, **val sequences** for retrieval eval:

| Directory | Source | Frames | Used in |
|-----------|--------|--------|---------|
| `reid_mot17_crops_train` | `mot17/train` | train-half (`1…L/2`) | §6 fine-tune (all identities) |
| `reid_mot17_crops_val` | `mot17/val` | val-half (`L/2+1…L`) | §7 retrieval (all val identities) |

Train and val are **different MOT sequences** (disjoint people). Val-half frames align with §9 tracking.

In [ ]:
train_stats = generate_mot_patches(
    MOT17_TRAIN,
    PATCH_DIR,
    split="train_half",
    min_visibility=0.3,
    min_side=16,
)
val_stats = generate_mot_patches(
    MOT17_VAL,
    PATCH_VAL_DIR,
    split="val_half",
    min_visibility=0.3,
    min_side=16,
)

print(f"Train crops: {train_stats.num_identities} identities, {train_stats.num_crops} crops")
print(f"Val crops:   {val_stats.num_identities} identities, {val_stats.num_crops} crops")

## 5. Retrieval eval split (val sequences)

Fine-tune on **every** train-sequence identity (§6). Build query/gallery from
**val-sequence** GT crops only — people the classifier never saw during training,
matching the same val domain as §9 tracking.

In [ ]:
val_retrieval_ids, query, gallery = build_retrieval_split(
    PATCH_VAL_DIR,
    queries_per_id=1,
)

print(f"Train identities (fine-tune): all {train_stats.num_identities} under {PATCH_DIR.name}")
print(f"Val retrieval identities:     {len(val_retrieval_ids)}  (query={len(query)}, gallery={len(gallery)})")

---
## 6. Fine-tune OSNet

Warm-start from the curated **OSNet / MSMT17** checkpoint (`pretrained=None`).
The classifier head is re-initialized for the MOT17 identity count.

Loss = label-smoothing CE + batch-hard triplet. Uses **all** train-sequence identities
from §4 (`include_identities=None`).

In [ ]:
config = TrainConfig(
    epochs=40,
    p=8,
    k=4,
    lr=3e-4,
    lr_milestones=(30, 35),
    freeze_backbone_epochs=3,
    triplet_weight=1.0,
    use_center=False,
)

result = train_reid(
    PATCH_DIR,
    config,
    pretrained=None,
    include_identities=None,
    output_dir=CHECKPOINT_DIR,
)

print(f"Checkpoint: {result.output_dir}")
print(f"Classes: {result.num_classes}  |  Final loss: {result.history[-1]['total']:.4f}")

In [ ]:
epochs = [row["epoch"] for row in result.history]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

for ax, key, title in zip(axes, ("ce", "triplet", "total"), ("Cross-entropy", "Triplet", "Total")):
    ax.plot(epochs, [row[key] for row in result.history], marker="o", ms=3)
    ax.set(xlabel="epoch", ylabel=key, title=title)
    ax.grid(True, alpha=0.25)

fig.suptitle("Fine-tune loss curves", y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Retrieval benchmark

Load both encoders and score the §5 **val-sequence** query/gallery split. Higher mAP /
Rank-1 → better generalization to unseen val identities (same domain as §9).

In [ ]:
baseline = ReIDModel.from_pretrained()
finetuned = ReIDModel.from_pretrained(str(CHECKPOINT_DIR))

for label, model in [
    ("OSNet (MSMT17 default)", baseline),
    ("OSNet (MOT17 finetuned)", finetuned),
]:
    m = ReIDEvaluator(model).evaluate(query, gallery, verbose=False).metrics
    print(f"{label:<28}  mAP={m.map:5.1f}%  Rank-1={m.rank1:5.1f}%")

---
## 8. Visualize embeddings

### 8a. Val-sequence GT crops

2D PCA on embeddings from **val-sequence GT crops** (same domain as §7). Same color = same identity.

**8b** repeats this on YOLOX detector crops. **8c** plots same-ID vs different-ID distance
histograms (BoT-SORT paper style).


In [ ]:
MAX_VIZ = 400
paths = list(query.image_paths) + list(gallery.image_paths)
pids = np.concatenate([query.pids, gallery.pids])

if len(paths) > MAX_VIZ:
    keep = np.sort(np.random.default_rng(0).choice(len(paths), MAX_VIZ, replace=False))
    paths, pids = [paths[i] for i in keep], pids[keep]

emb_msmt = baseline.extract_features_from_paths(paths, batch_size=64, normalize=True)
emb_mot = finetuned.extract_features_from_paths(paths, batch_size=64, normalize=True)

plot_pca_side_by_side(
    emb_msmt,
    emb_mot,
    pids,
    left_title="OSNet (MSMT17 default)",
    right_title="OSNet (MOT17 finetuned)",
    suptitle=f"Val-sequence GT crops — {len(paths)} points, {len(np.unique(pids))} identities",
)

### 8b. YOLOX detector crops

Same PCA comparison on **YOLOX detections** matched to val-half GT by IoU — the crop
domain BoT-SORT actually sees at inference.

In [ ]:
SEQ = "MOT17-02-FRCNN"

emb_msmt, gt_ids = collect_yolox_gt_embeddings(baseline, [SEQ], frame_stride=5)
emb_mot, _ = collect_yolox_gt_embeddings(finetuned, [SEQ], frame_stride=5)

if len(gt_ids) > MAX_VIZ:
    keep = np.sort(np.random.default_rng(0).choice(len(gt_ids), MAX_VIZ, replace=False))
    emb_msmt, emb_mot, gt_ids = emb_msmt[keep], emb_mot[keep], gt_ids[keep]

plot_pca_side_by_side(
    emb_msmt,
    emb_mot,
    gt_ids,
    left_title="OSNet (MSMT17 default)",
    right_title="OSNet (MOT17 finetuned)",
    suptitle=f"YOLOX crops — {SEQ} ({len(gt_ids)} points, {len(np.unique(gt_ids))} identities)",
)

### 8c. Same-ID vs different-ID distance distributions

Classic ReID diagnostic (see BoT-SORT paper): histogram of **cosine distance** between
detector-crop embeddings from the **same GT track** vs **different GT tracks**.

A good encoder pushes same-ID distances left and separates the two distributions.
We sample pairs from YOLOX crops across all val sequences.

The dashed **θ_emb** line is the appearance gate used in §9 (`d_app < θ_emb`).
Tune `EMB_DIST_MSMT17` / `EMB_DIST_MOT17` below after inspecting the overlap, then re-run §9.


In [ ]:
# Collect tracking-domain embeddings once per encoder (all val sequences).
emb_msmt, gt_ids = collect_yolox_gt_embeddings(
    baseline,
    VAL_SEQS,
    frame_stride=5,
    max_frames_per_seq=30,
)
emb_mot, _ = collect_yolox_gt_embeddings(
    finetuned,
    VAL_SEQS,
    frame_stride=5,
    max_frames_per_seq=30,
)

intra_msmt, inter_msmt = sample_intra_inter_cosine_distances(emb_msmt, gt_ids, seed=0)
intra_mot, inter_mot = sample_intra_inter_cosine_distances(emb_mot, gt_ids, seed=0)

EMB_DIST_MSMT17 = suggest_emb_dist_threshold(intra_msmt, inter_msmt)
print("msmt dis threshold", EMB_DIST_MSMT17)
EMB_DIST_MOT17 = suggest_emb_dist_threshold(intra_mot, inter_mot)
print("mot dis threshold", EMB_DIST_MOT17)
# Override after inspecting the histogram (cosine distance; BoT-SORT paper default is 0.25):
# EMB_DIST_MSMT17 = 0.40
# EMB_DIST_MOT17 = 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=True)
plot_intra_inter_distance_histogram(
    intra_msmt,
    inter_msmt,
    title="OSNet (MSMT17 default)",
    ax=axes[0],
    emb_dist_threshold=EMB_DIST_MSMT17,
)
plot_intra_inter_distance_histogram(
    intra_mot,
    inter_mot,
    title="OSNet (MOT17 finetuned)",
    ax=axes[1],
    emb_dist_threshold=EMB_DIST_MOT17,
)
fig.suptitle(
    f"YOLOX crop cosine distances — {len(gt_ids)} embeddings, {len(np.unique(gt_ids))} GT ids",
    y=1.02,
)
plt.tight_layout()
plt.show()

for label, intra, inter, theta in [
    ("MSMT17 default", intra_msmt, inter_msmt, EMB_DIST_MSMT17),
    ("MOT17 finetuned", intra_mot, inter_mot, EMB_DIST_MOT17),
]:
    print(
        f"{label}: same-id mean={intra.mean():.3f}  diff-id mean={inter.mean():.3f}  "
        f"gap={inter.mean() - intra.mean():.3f}  θ_emb={theta:.3f} (cos sim > {1 - theta:.2f})"
    )

In [ ]:
# EMB_DIST_MSMT17 = 0.5
# EMB_DIST_MOT17 = 0.4

---
## 9. Tracking benchmark

Run **BoT-SORT + CMC** on all seven val sequences. GT and frames come from `mot17/val/`;
detections come from the YOLOX zip (§3b).

We compare three configs: no ReID, MSMT17 ReID (baseline encoder), and MOT17-finetuned ReID.

In [ ]:
from trackers import BoTSORTTracker
from trackers.eval import evaluate_mot_sequences
from trackers.eval.results import BenchmarkResult
from trackers.io.mot import _MOTOutput

# Set False to reuse cached preds under outputs/trackers_reid_outputs/
RERUN = {
    "botsort_baseline": False,
    "botsort_reid_msmt17": True,
    "botsort_reid_mot17": True,
}

TRACK_OUT.mkdir(parents=True, exist_ok=True)
seqmap_path = TRACK_OUT / "MOT17-val.txt"
seqmap_path.write_text("name\n" + "\n".join(VAL_SEQS) + "\n")

In [ ]:
def track_val_split(name: str, tracker_factory) -> BenchmarkResult:
    pred_dir = TRACK_OUT / name / "preds"
    cache = TRACK_OUT / name / "eval_results.json"

    if not RERUN.get(name, True) and cache.is_file():
        print("  (cached)")
        return BenchmarkResult.load(cache)

    pred_dir.mkdir(parents=True, exist_ok=True)
    for seq in VAL_SEQS:
        det_by_frame = load_val_dets(yolox_det_path(seq))
        frame_paths = sorted((MOT17_VAL / seq / "img1").glob("*.jpg"))
        tracker = tracker_factory()

        with _MOTOutput(pred_dir / f"{seq}.txt") as writer:
            for frame_idx, frame_path in enumerate(frame_paths, start=1):
                dets = det_by_frame.get(frame_idx, sv.Detections.empty())
                tracked = tracker.update(dets, cv2.imread(str(frame_path)))
                if tracked.tracker_id is not None:
                    tracked = tracked[tracked.tracker_id != -1]
                writer.write(frame_idx, tracked)

    result = evaluate_mot_sequences(
        gt_dir=MOT17_VAL,
        tracker_dir=pred_dir,
        seqmap=seqmap_path,
        metrics=["CLEAR", "HOTA", "Identity"],
    )
    result.save(cache)
    return result


experiments = [
    ("BoT-SORT (baseline)", "botsort_baseline", lambda: BoTSORTTracker(enable_cmc=True, reid_model=None)),
    (
        "BoT-SORT + ReID (MSMT17)",
        "botsort_reid_msmt17",
        lambda: BoTSORTTracker(
            enable_cmc=True,
            reid_model=baseline,
            reid_ema_alpha=0.9,
            appearance_threshold=EMB_DIST_MSMT17,
        ),
    ),
    (
        "BoT-SORT + ReID (MOT17)",
        "botsort_reid_mot17",
        lambda: BoTSORTTracker(
            enable_cmc=True,
            reid_model=finetuned,
            reid_ema_alpha=0.9,
            appearance_threshold=EMB_DIST_MOT17,
        ),
    ),
]

results = {}
for label, name, factory in experiments:
    print(label)
    result = track_val_split(name, factory)
    results[name] = result
    agg = result.aggregate
    print(
        f"  HOTA={agg.HOTA.HOTA * 100:6.2f}  "
        f"MOTA={agg.CLEAR.MOTA * 100:6.2f}  "
        f"IDF1={agg.Identity.IDF1 * 100:6.2f}  "
        f"IDSW={agg.CLEAR.IDSW}"
    )

msmt, mot = results["botsort_reid_msmt17"].aggregate, results["botsort_reid_mot17"].aggregate
print("\nMOT17 finetuned − MSMT17 (ReID configs):")
print(f"  ΔHOTA {100 * (mot.HOTA.HOTA - msmt.HOTA.HOTA):+6.2f}")
print(f"  ΔIDF1 {100 * (mot.Identity.IDF1 - msmt.Identity.IDF1):+6.2f}")